# Project 1 : Information Retrieval as a recommender system
## Thom CHHUN
## Ali DABALE ALI

## Corpus : TripAdvisor
- ### Main idea : based on my past reviews, how to recommend the most similar trips?
- ### Hypothesis : similar places/experiences are described with similar words

## Input/Output Model
- ### Input : a review or set of review on a specific place
- ### Output : most similar place
- ### Model relies only on reviews and not on other metadatas

In [7]:
# ==============================
# Imports
# ==============================

import numpy as np
import pandas as pd
import os
import pickle
import re

# NLP - NLTK
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Gensim
import gensim.downloader

# BM25
from rank_bm25 import BM25Okapi

# ==============================
# Download NLTK Resources
# ==============================

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Thom\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Thom\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Thom\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
from evaluation import (
    evaluation_level1,
    evaluation_level2,
    evaluation_level3,
    evaluation_level4,
    evaluation_level5,
    evaluation_level6,
    evaluation_level7,
    evaluation_level8
)

from evaluation import preprocess_text

In [9]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [10]:
reviews = pd.read_csv("data/reviews_final.csv")
tripadvisor = pd.read_csv("data/tripadvisor_final.csv")

créer une application streamlit qui permet de montrer ce qu'il se passe, input choisi directement parmis le test dataset, il y a pas vraiment de modèle, les données c'est l'ensemble, c'est complétement fictif cette histoire de split. Il faudrait expliquer pourquoi certain resultats remonte, montrer, surligner les mots qui ont été utilisé pour matcher les lieux les plus pertinents.

Modèle d'embedding beaucoup plus compliqué, créer un petit algo pour mettre en évidence la similitude pour mettre en évidence certains mots spécifique.

## 0) Baseline : BM25

In [ ]:
# ==============================
# Appliquer le prétraitement à toutes les reviews
# ==============================
reviews['cleaned_review'] = reviews['review'].apply(preprocess_text)
# ==============================
# Regrouper par lieu
# ==============================
grouped = reviews.groupby('idplace')['cleaned_review'].apply(lambda x: " ".join(x)).reset_index()
#grouped.to_csv("reviews.csv", index=False, encoding="utf-8")

# ==============================
# Split 50/50 des lieux pour evaluation
# ==============================
X_train00, X_test00 = train_test_split(grouped, test_size=0.1, random_state=42, shuffle=True)

#X_train00.to_csv("data/X_train.csv", index=False, encoding="utf-8")
#X_test00.to_csv("data/X_test.csv", index=False, encoding="utf-8")

# ==============================
# Préparer corpus tokenisé
# ==============================
tokenized_train = [
    doc.split() for doc in X_train00['cleaned_review']
]

bm25 = BM25Okapi(tokenized_train)

# ==============================
# Générer matrice de similarité BM25
# ==============================
similarity_matrix_m00 = []

for query in X_test00['cleaned_review']:
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    similarity_matrix_m00.append(scores)

similarity_matrix_m00 = np.array(similarity_matrix_m00)

In [12]:
tripadvisor_meta = tripadvisor.set_index("id")

evaluation_level1_m00 = evaluation_level1(X_test00, X_train00, similarity_matrix_m00, tripadvisor, top_k=5)
print(f"Precision Level 1 (Normalized Top-5 Type Score) : {evaluation_level1_m00:.4f}")

evaluation_level2_m00 = evaluation_level2( X_test00, X_train00, similarity_matrix_m00, tripadvisor_meta, top_k=5)
print(f"Precision Level 2 (Normalized Top-5 Metadata Score) : {evaluation_level2_m00:.4f}")

evaluation_level3_m00 = evaluation_level3(X_test00, X_train00, similarity_matrix_m00, tripadvisor)
print(f"Precision Level 3 (Top-1 Type Accuracy) : {evaluation_level3_m00:.4f}")

evaluation_level4_m00 = evaluation_level4(X_test00, X_train00, similarity_matrix_m00, tripadvisor_meta)
print(f"Precision Level 4 (Top-1 Metadata Accuracy) : {evaluation_level4_m00:.4f}")

evaluation_level5_m00 = evaluation_level5(X_test00, X_train00, similarity_matrix_m00, tripadvisor, top_k=5)
print(f"Precision Level 5 (Top-5 Type Recall) : {evaluation_level5_m00:.4f}")

evaluation_level6_m00 = evaluation_level6(X_test00, X_train00, similarity_matrix_m00, tripadvisor_meta, top_k=5)
print(f"Precision Level 6 (Top-5 Metadata Recall) : {evaluation_level6_m00:.4f}")

evaluation_level7_m00 = evaluation_level7(X_test00, X_train00, similarity_matrix_m00, tripadvisor)
print(f"Precision Level 7 (Mean Rank of First Relevant Type Match) : {evaluation_level7_m00:.2f}")

evaluation_level8_m00 = evaluation_level8(X_test00, X_train00, similarity_matrix_m00, tripadvisor_meta)
print(f"Precision Level 8 (Mean Rank of First Relevant Metadata Match) : {evaluation_level8_m00:.2f}")

Precision Level 1 (Normalized Top-5 Type Score) : 0.7671
Precision Level 2 (Normalized Top-5 Metadata Score) : 0.5908
Precision Level 3 (Top-1 Type Accuracy) : 0.7882
Precision Level 4 (Top-1 Metadata Accuracy) : 0.6202
Precision Level 5 (Top-5 Type Recall) : 0.9647
Precision Level 6 (Top-5 Metadata Recall) : 0.5765
Precision Level 7 (Mean Rank of First Relevant Type Match) : 1.60
Precision Level 8 (Mean Rank of First Relevant Metadata Match) : 59.55


## 1) tokenization, lemmatisation, regroupement par lieu, TF-IDF et similarité cosine TF-IDF Bag of Words representation

In [13]:
# ==============================
# Appliquer le prétraitement à toutes les reviews
# ==============================
reviews['cleaned_review'] = reviews['review'].apply(preprocess_text)

# ==============================
# Regrouper par lieu
# ==============================
grouped = reviews.groupby('idplace')['cleaned_review'].apply(lambda x: " ".join(x)).reset_index()

# ==============================
# Split 50/50 des lieux pour evaluation
# ==============================
X_train01, X_test01 = train_test_split(grouped, test_size=0.1, random_state=42, shuffle=True)

# TF-IDF uniquement sur train
vectorizer = TfidfVectorizer(max_features=1000)
tfidf_train = vectorizer.fit_transform(X_train01['cleaned_review'])

# Transformer test avec le même vocabulaire
tfidf_test = vectorizer.transform(X_test01['cleaned_review'])

# ==============================
# Calcul de similarité cosine (test vs train)
# ==============================
similarity_matrix_m01 = cosine_similarity(tfidf_test, tfidf_train)  # chaque test vs tous les train

In [14]:
def recommend_similar_place_tfidf(idplace_query, top_k=5, top_n_words=5):

    if idplace_query not in X_test01['idplace'].values:
        return f"L'idplace {idplace_query} n'est pas dans X_test."

    # Reset index
    X_test_reset = X_test01.reset_index(drop=True)
    query_pos = X_test_reset.index[X_test_reset['idplace'] == idplace_query][0]

    sims = similarity_matrix_m01[query_pos]

    # Trier les similarités décroissantes
    top_indices_sim = np.argsort(sims)[::-1][:top_k]

    query_vector = tfidf_test[query_pos].toarray()[0]
    feature_names = np.array(vectorizer.get_feature_names_out())

    print(f"Query Place: {idplace_query}\n")

    for rank, idx in enumerate(top_indices_sim, start=1):

        similar_id = X_train01.iloc[idx]['idplace']
        score = sims[idx]

        similar_vector = tfidf_train[idx].toarray()[0]

        # Contribution commune
        contribution = query_vector * similar_vector
        top_word_indices = contribution.argsort()[::-1][:top_n_words]

        print(f"Recommandation {rank}: {similar_id}")
        print(f"Similarity Score: {score:.3f}")
        print("Top Contributing Words:")

        for word_idx in top_word_indices:
            if contribution[word_idx] > 0:
                word = feature_names[word_idx]
                sc = contribution[word_idx]
                print(f"  - {word} (contribution: {sc:.4f})")
        print()

    return [int(X_train01.iloc[idx]['idplace']) for idx in top_indices_sim]

#id_to_test = 3464234  # remplace par l'id que tu veux
#print(recommend_similar_place_tfidf_explain(id_to_test))
id_to_test = 1725986
print(recommend_similar_place_tfidf(id_to_test))

Query Place: 1725986

Recommandation 1: 5924082
Similarity Score: 0.892
Top Contributing Words:
  - crepe (contribution: 0.0890)
  - waiter (contribution: 0.0699)
  - notre (contribution: 0.0453)
  - dame (contribution: 0.0452)
  - food (contribution: 0.0440)

Recommandation 2: 1755379
Similarity Score: 0.891
Top Contributing Words:
  - food (contribution: 0.0655)
  - waiter (contribution: 0.0606)
  - service (contribution: 0.0420)
  - good (contribution: 0.0371)
  - place (contribution: 0.0290)

Recommandation 3: 2087229
Similarity Score: 0.881
Top Contributing Words:
  - waiter (contribution: 0.0700)
  - food (contribution: 0.0653)
  - dame (contribution: 0.0467)
  - crepe (contribution: 0.0455)
  - notre (contribution: 0.0441)

Recommandation 4: 2239316
Similarity Score: 0.860
Top Contributing Words:
  - waiter (contribution: 0.0644)
  - dame (contribution: 0.0479)
  - notre (contribution: 0.0459)
  - food (contribution: 0.0449)
  - order (contribution: 0.0411)

Recommandation 5: 23

In [15]:
#ids_to_keep = [  695212 , 695185,799588,1309236,3464234,1500304]
ids_to_keep = [1725986 , 5924082]
tripadvisor_test = tripadvisor[tripadvisor['id'].isin(ids_to_keep)]
tripadvisor_test

,id,nom,url,website,rating,nbAvis,latitude,longitude,typeR,adresse,priceRange,activiteType,activiteSubType,activiteCategory,restaurantType,restaurantCategory,restaurantCuisine,restaurantDietaryRestrictions,hotelType,hotelpriceRange
261,1725986,Le Soleil D'Or,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d1725986-Reviews-Le_Soleil_D_Or-Paris_Ile_de_France.html,http://www.facebook.com/pages/Le-Soleil-Dor/237135486415057,3.134947,387,48.854130,2.345115,R,"15 boulevard du Palais, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, Cafe, European",NaN,NaN,NaN
515,5924082,Brasserie Esmeralda,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d5924082-Reviews-Brasserie_Esmeralda-Paris_Ile_de_France.html,NaN,2.740275,379,48.852695,2.352112,R,"2 rue du Cloitre Notre Dame, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, European",NaN,NaN,NaN


In [16]:
tripadvisor_meta = tripadvisor.set_index("id")

evaluation_level1_m01 = evaluation_level1(X_test01, X_train01, similarity_matrix_m01, tripadvisor, top_k=5)
print(f"Precision Level 1 (Normalized Top-5 Type Score) : {evaluation_level1_m01:.4f}")

evaluation_level2_m01 = evaluation_level2(X_test01, X_train01, similarity_matrix_m01, tripadvisor_meta, top_k=5)
print(f"Precision Level 2 (Normalized Top-5 Metadata Score) : {evaluation_level2_m01:.4f}")

evaluation_level3_m01 = evaluation_level3(X_test01, X_train01, similarity_matrix_m01, tripadvisor)
print(f"Precision Level 3 (Top-1 Type Accuracy) : {evaluation_level3_m01:.4f}")

evaluation_level4_m01 = evaluation_level4(X_test01, X_train01, similarity_matrix_m01, tripadvisor_meta)
print(f"Precision Level 4 (Top-1 Metadata Accuracy) : {evaluation_level4_m01:.4f}")

evaluation_level5_m01 = evaluation_level5(X_test01, X_train01, similarity_matrix_m01, tripadvisor, top_k=5)
print(f"Precision Level 5 (Top-5 Type Recall) : {evaluation_level5_m01:.4f}")

evaluation_level6_m01 = evaluation_level6(X_test01, X_train01, similarity_matrix_m01, tripadvisor_meta, top_k=5)
print(f"Precision Level 6 (Top-5 Metadata Recall) : {evaluation_level6_m01:.4f}")


evaluation_level7_m01 = evaluation_level7(X_test01, X_train01, similarity_matrix_m01, tripadvisor)
print(f"Precision Level 7 (Mean Rank of First Relevant Type Match) : {evaluation_level7_m01:.2f}")

evaluation_level8_m01 = evaluation_level8(X_test01, X_train01, similarity_matrix_m01, tripadvisor_meta)
print(f"Precision Level 8 (Mean Rank of First Relevant Metadata Match) : {evaluation_level8_m01:.2f}")

Precision Level 1 (Normalized Top-5 Type Score) : 0.8400
Precision Level 2 (Normalized Top-5 Metadata Score) : 0.6809
Precision Level 3 (Top-1 Type Accuracy) : 0.8824
Precision Level 4 (Top-1 Metadata Accuracy) : 0.7504
Precision Level 5 (Top-5 Type Recall) : 0.9765
Precision Level 6 (Top-5 Metadata Recall) : 0.6941
Precision Level 7 (Mean Rank of First Relevant Type Match) : 1.60
Precision Level 8 (Mean Rank of First Relevant Metadata Match) : 42.88


Le score de similarité cosine entre deux lieux dans la méthode TF-IDF ne regarde pas juste le nombre de mots communs, il regarde la pondération de ces mots communs dans le contexte global.

## Word Embeddings + moyenne / Pretrained Word Embedding + Document averaging

In [17]:
#!pip install gensim
import gensim.downloader
import pickle
import os

class gensim_interface:

    """
    #List of possible embeddings:

    ['fasttext-wiki-news-subwords-300',
     'conceptnet-numberbatch-17-06-300',
     'word2vec-ruscorpora-300',
     'word2vec-google-news-300',
     'glove-wiki-gigaword-50',
     'glove-wiki-gigaword-100',
     'glove-wiki-gigaword-200',
     'glove-wiki-gigaword-300',
     'glove-twitter-25',
     'glove-twitter-50',
     'glove-twitter-100',
     'glove-twitter-200',
     '__testing_word2vec-matrix-synopsis']
    """

    #load and serialize the embeddings in order to accelerate next loads
    #input: name of the embedding wanted (example : 'glove-twitter-25')
    #output: the vector embeddings
    def embeddingPreparation(self,embeddingName):
        self.embeddingVectors = gensim.downloader.load(embeddingName)
        with open(embeddingName + ".vecs","wb") as fd:
            pickle.dump(self.embeddingVectors,fd)
        #return self.embeddingVectors

    #load already serialized embeddings (call of embeddingPreparation() needed before)
    #input: name of the embedding wanted (example : 'glove-twitter-25')
    #output: the vector embeddings
    def loadPreparedEmbedding(self,embeddingName):
        with open(embeddingName + ".vecs","rb") as fd:
            self.embeddingVectors = pickle.load(fd)
        #return self.embeddingVectors

    #input: word or id of the word
    #output: True if word in the embedding list, False otherwise
    def isVec(self,word):
        if word in self.embeddingVectors:
            return True
        else:
            return False

    #input: word(string) or word id(int)
    #output: vector embedding
    def getVec(self,word):
        return self.embeddingVectors[word]

    #input: word(string)
    #output: word id(int)
    def getId(self,word):
        return self.embeddingVectors.key_to_index[word]

    #input: word id(int)
    #output: word(string)
    def getWord(self,id):
        return self.embeddingVectors.index_to_key[id]

    #output : list of all words covered by the embedding
    def getVocabList(self):
        return self.embeddingVectors.index_to_key

    #output : return vocabulary dictionary associating word and idWords (key=word, value=idWord)
    def getVocabDic(self):
        return self.embeddingVectors.key_to_index

    #output : number of words covered by the embedding
    def getLenVocab(self):
        return len(self.embeddingVectors.index_to_key)

    #output :  number of dimensions of the embedding
    def nbDims(self):
        return self.embeddingVectors.vector_size

    #output : list of all embeddings names availables on gensim
    def getAvailableEmbeddings(self):
        return list(gensim.downloader.info()['models'].keys())

    #input : word , number of most similar words wanted
    #output : list of n most similar words
    def getMostSimilar(self,word,n):
        ms = self.embeddingVectors.most_similar(word,topn=n)
        neighbours = [elem[0] for elem in ms]
        return neighbours

    ########################################################
    ########################################################
    ########################################################

    def __init__(self,embeddingName):
        #test if file exists
        fileName = embeddingName + ".vecs"
        if os.path.isfile(fileName):
            print("loading embeddings...")
            self.loadPreparedEmbedding(embeddingName)
        else:
            print("embedding preparation...")
            self.embeddingPreparation(embeddingName)
        self.vectors = self.embeddingVectors.vectors

In [30]:
# ===================================
# Charger embedding
# ===================================
emb = gensim_interface('glove-wiki-gigaword-100')
#emb = gensim_interface('glove-wiki-gigaword-200')
#100 a de meilleur résultat que 300

# ===================================
# Regrouper reviews par lieu
# ===================================
reviews['cleaned_review'] = reviews['review'].apply(preprocess_text)
grouped = reviews.groupby('idplace')['cleaned_review'].apply(
    lambda x: " ".join(x)
).reset_index()

# ===================================
# Split Train / Test (50% recommandé en évaluation IR)
# ===================================
X_train02, X_test02 = train_test_split(
    grouped,
    test_size=0.1,
    random_state=42
)

# IMPORTANT Reset index pour utiliser similarity matrix positionnelle
X_train02 = X_train02.reset_index(drop=True)
X_test02 = X_test02.reset_index(drop=True)

# ===================================
# Fonction embedding document
# ===================================
def get_document_embedding(text, emb):
    words = text.split()
    vectors = []
    for word in words:
        if emb.isVec(word):
            vectors.append(emb.getVec(word))

    if len(vectors) == 0:
        return np.zeros(emb.nbDims())

    return np.mean(vectors, axis=0)

# ===================================
# Construire embeddings TRAIN
# ===================================
train_embeddings = []
for text in X_train02['cleaned_review']:
    train_embeddings.append(
        get_document_embedding(text, emb)
    )

train_embeddings = np.array(train_embeddings)

# ===================================
# Construire embeddings TEST
# ===================================
test_embeddings = []
for text in X_test02['cleaned_review']:
    test_embeddings.append(
        get_document_embedding(text, emb)
    )

test_embeddings = np.array(test_embeddings)

# ===================================
# Similarité cosine Test vs Train
# ===================================
similarity_matrix_m02 = cosine_similarity(
    test_embeddings,
    train_embeddings
)

loading embeddings...


In [31]:
def recommend_similar_place_embedding(idplace_query, top_k=5, top_n_dims=5):
    # Reset propre
    X_test_reset = X_test02.reset_index(drop=True)
    X_train_reset = X_train02.reset_index(drop=True)
    if idplace_query not in X_test_reset['idplace'].values:
        return "Idplace not found in test set"

    # Position query
    query_idx = X_test_reset.index[
        X_test_reset['idplace'] == int(idplace_query)
    ][0]
    # Similarités
    sims = similarity_matrix_m02[query_idx].copy()
    # Top-K indices triés décroissants
    top_indices_sim = np.argsort(sims)[::-1][:top_k]
    query_vec = test_embeddings[query_idx]
    print(f"Query Place: {idplace_query}\n")
    for rank, idx in enumerate(top_indices_sim, start=1):
        rec_id = X_train_reset.iloc[idx]['idplace']
        score = sims[idx]
        similar_vec = train_embeddings[idx]
        # Contribution dimensionnelle
        contribution = query_vec * similar_vec
        top_dims = contribution.argsort()[::-1][:top_n_dims]
        print(f"Recommandation {rank}: {rec_id}")
        print(f"Similarity Score: {score:.3f}")
        print("Top Contributing Words:")
        for dim in top_dims:
            print(
                f"  - Dimension {dim} "
                f"(contribution: {contribution[dim]:.6f})"
            )
        print()
    return [int(X_train02.iloc[idx]['idplace']) for idx in top_indices_sim]

id_to_test = 1725986
print(recommend_similar_place_embedding(id_to_test))

Query Place: 1725986

Recommandation 1: 5924082
Similarity Score: 0.999
Top Contributing Words:
  - Dimension 55 (contribution: 2.801872)
  - Dimension 58 (contribution: 1.203722)
  - Dimension 84 (contribution: 0.858322)
  - Dimension 53 (contribution: 0.479581)
  - Dimension 49 (contribution: 0.364385)

Recommandation 2: 2337904
Similarity Score: 0.999
Top Contributing Words:
  - Dimension 55 (contribution: 2.855095)
  - Dimension 58 (contribution: 1.274223)
  - Dimension 84 (contribution: 0.877791)
  - Dimension 53 (contribution: 0.476234)
  - Dimension 49 (contribution: 0.383227)

Recommandation 3: 2184631
Similarity Score: 0.998
Top Contributing Words:
  - Dimension 55 (contribution: 2.806291)
  - Dimension 58 (contribution: 1.207279)
  - Dimension 84 (contribution: 0.847144)
  - Dimension 53 (contribution: 0.471281)
  - Dimension 49 (contribution: 0.374648)

Recommandation 4: 2087229
Similarity Score: 0.998
Top Contributing Words:
  - Dimension 55 (contribution: 2.803429)
  - Dim

In [20]:
ids_to_keep = [1725986, 5924082]
tripadvisor_test = tripadvisor[tripadvisor['id'].isin(ids_to_keep)]
tripadvisor_test

,id,nom,url,website,rating,nbAvis,latitude,longitude,typeR,adresse,priceRange,activiteType,activiteSubType,activiteCategory,restaurantType,restaurantCategory,restaurantCuisine,restaurantDietaryRestrictions,hotelType,hotelpriceRange
261,1725986,Le Soleil D'Or,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d1725986-Reviews-Le_Soleil_D_Or-Paris_Ile_de_France.html,http://www.facebook.com/pages/Le-Soleil-Dor/237135486415057,3.134947,387,48.854130,2.345115,R,"15 boulevard du Palais, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, Cafe, European",NaN,NaN,NaN
515,5924082,Brasserie Esmeralda,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d5924082-Reviews-Brasserie_Esmeralda-Paris_Ile_de_France.html,NaN,2.740275,379,48.852695,2.352112,R,"2 rue du Cloitre Notre Dame, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, European",NaN,NaN,NaN


In [32]:
tripadvisor_meta = tripadvisor.set_index("id")

evaluation_level1_m02 = evaluation_level1(X_test02, X_train02, similarity_matrix_m02, tripadvisor, top_k=5)
print(f"Precision Level 1 (Normalized Top-5 Type Score) : {evaluation_level1_m02:.4f}")

evaluation_level2_m02 = evaluation_level2(X_test02, X_train02, similarity_matrix_m02, tripadvisor_meta, top_k=5)
print(f"Precision Level 2 (Normalized Top-5 Metadata Score) : {evaluation_level2_m02:.4f}")

evaluation_level3_m02 = evaluation_level3(X_test02, X_train02, similarity_matrix_m02, tripadvisor)
print(f"Precision Level 3 (Top-1 Type Accuracy) : {evaluation_level3_m02:.4f}")

evaluation_level4_m02 = evaluation_level4(X_test02, X_train02, similarity_matrix_m02, tripadvisor_meta)
print(f"Precision Level 4 (Top-1 Metadata Accuracy) : {evaluation_level4_m02:.4f}")

evaluation_level5_m02 = evaluation_level5(X_test02, X_train02, similarity_matrix_m02, tripadvisor, top_k=5)
print(f"Precision Level 5 (Top-5 Type Recall) : {evaluation_level5_m02:.4f}")

evaluation_level6_m02 = evaluation_level6(X_test02, X_train02, similarity_matrix_m02, tripadvisor_meta, top_k=5)
print(f"Precision Level 6 (Top-5 Metadata Recall) : {evaluation_level6_m02:.4f}")

evaluation_level7_m02 = evaluation_level7(X_test02, X_train02, similarity_matrix_m02, tripadvisor)
print(f"Precision Level 7 (Mean Rank of First Relevant Type Match) : {evaluation_level7_m02:.2f}")

evaluation_level8_m02 = evaluation_level8(X_test02, X_train02, similarity_matrix_m02, tripadvisor_meta)
print(f"Precision Level 8 (Mean Rank of First Relevant Metadata Match) : {evaluation_level8_m02:.2f}")

Precision Level 1 (Normalized Top-5 Type Score) : 0.8659
Precision Level 2 (Normalized Top-5 Metadata Score) : 0.6993
Precision Level 3 (Top-1 Type Accuracy) : 0.8941
Precision Level 4 (Top-1 Metadata Accuracy) : 0.7659
Precision Level 5 (Top-5 Type Recall) : 0.9765
Precision Level 6 (Top-5 Metadata Recall) : 0.7059
Precision Level 7 (Mean Rank of First Relevant Type Match) : 1.40
Precision Level 8 (Mean Rank of First Relevant Metadata Match) : 51.62


Mots → points dans l’espace

Lieu → centre de gravité de ses points

Score → combien deux centres de gravité “regardent” dans la même direction

## TF-IDF + retrieval + Precision@K

In [22]:
# ===================================
# Mapping metadata
# ===================================

id_to_typeR = tripadvisor.set_index('id')['typeR'].to_dict()
# Ajouter typeR dans grouped dataset
reviews['cleaned_review'] = reviews['review'].apply(preprocess_text)
grouped = reviews.groupby('idplace')['cleaned_review'].apply(lambda x: " ".join(x)).reset_index()
grouped['typeR'] = grouped['idplace'].map(id_to_typeR)

# ===================================
# Train / Test Split
# ===================================
X_train03, X_test03 = train_test_split(
    grouped,
    test_size=0.1,
    random_state=42
)
id_to_typeR = tripadvisor.set_index('id')['typeR'].to_dict()
X_train03['typeR'] = X_train03['idplace'].map(id_to_typeR)
X_test03['typeR'] = X_test03['idplace'].map(id_to_typeR)
X_train03 = X_train03.reset_index(drop=True)
X_test03 = X_test03.reset_index(drop=True)

# ===================================
# TF-IDF Representation (TRAIN index)
# ===================================
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000
)

X_train_tfidf = vectorizer.fit_transform(
    X_train03['cleaned_review']
)

X_test_tfidf = vectorizer.transform(
    X_test03['cleaned_review']
)

# ===================================
# Cosine Similarity Retrieval
# ===================================
similarity_matrix_m03 = cosine_similarity(
    X_test_tfidf,
    X_train_tfidf
)

# ===================================
# Top-K retrieval function
# ===================================
def topk_indices(sim_matrix, k):
    k = min(k, sim_matrix.shape[1])
    part = np.argpartition(sim_matrix, -k, axis=1)[:, -k:]
    row = np.arange(sim_matrix.shape[0])[:, None]
    order = np.argsort(sim_matrix[row, part], axis=1)[:, ::-1]
    return part[row, order]

# ===================================
# Precision@K Evaluation
# ===================================
def precision_at_k(retrieved_idx, train_labels, query_labels):
    k = retrieved_idx.shape[1]
    retrieved_labels = train_labels[retrieved_idx]
    matches = (retrieved_labels == query_labels[:, None]).mean(axis=1)
    return float(matches.mean())

# ===================================
# Build category arrays
# ===================================
train_categories = X_train03['typeR'].fillna("UNKNOWN").values
test_categories = X_test03['typeR'].fillna("UNKNOWN").values

# ===================================
# Evaluation
# ===================================
K = 10
topk = topk_indices(similarity_matrix_m03, K)
precision_level03 = precision_at_k(
    topk,
    train_categories,
    test_categories
)

# ===================================
# Detailed evaluation by category
# ===================================
pos_mask = (test_categories == 'H')
rest_mask = (test_categories == 'R')
p_pos = precision_at_k(topk[pos_mask], train_categories, test_categories[pos_mask])
p_rest = precision_at_k(topk[rest_mask], train_categories, test_categories[rest_mask])

print(f"\nPrecision@{K} Method 3 (TF-IDF Retrieval) : {precision_level03:.4f}")
print(f"Precision Pos Queries : {p_pos:.4f}")
print(f"Precision Restaurant Queries : {p_rest:.4f}")


Precision@10 Method 3 (TF-IDF Retrieval) : 0.8353
Precision Pos Queries : 1.0000
Precision Restaurant Queries : 0.9677


In [23]:
import numpy as np

def recommend_similar_place_retrieval(idplace_query, top_k=5, top_words=5):
    # Trouver la ligne du query dans le test set
    query_row = X_test03[X_test03['idplace'] == int(idplace_query)]
    if len(query_row) == 0:
        return "Idplace non trouvé dans le test set"

    query_idx = query_row.index[0]
    # Similarités
    sims = similarity_matrix_m03[query_idx].copy()
    # Trier les similarités décroissantes
    top_indices_sim = np.argsort(sims)[::-1][:top_k]
    # Vecteur query
    q_vec = X_test_tfidf[query_idx]
    feature_names = vectorizer.get_feature_names_out()
    print(f"Query Place: {idplace_query}\n")
    for rank, idx in enumerate(top_indices_sim, start=1):
        rec_id = X_train03.iloc[idx]['idplace']
        score = sims[idx]
        # Produit terme à terme
        d_vec = X_train_tfidf[idx]
        contribution = q_vec.multiply(d_vec)
        contribution_array = contribution.toarray().flatten()
        # Top mots
        top_word_indices = np.argsort(contribution_array)[::-1][:top_words]
        print(f"Recommendation {rank}: {rec_id}")
        print(f"Similarity Score: {score:.3f}")
        print("Top Contributing Words:")

        for word_idx in top_word_indices:
            if contribution_array[word_idx] > 0:
                print(f"- {feature_names[word_idx]} "
                      f"(contribution: {contribution_array[word_idx]:.4f})")

        print()
    return [int(X_train03.iloc[idx]['idplace']) for idx in top_indices_sim]
        
id_to_test = 1725986
print(recommend_similar_place_retrieval(id_to_test))

Query Place: 1725986

Recommendation 1: 5924082
Similarity Score: 0.886
Top Contributing Words:
- crepe (contribution: 0.0960)
- waiter (contribution: 0.0754)
- notre (contribution: 0.0489)
- dame (contribution: 0.0487)
- food (contribution: 0.0475)

Recommendation 2: 2087229
Similarity Score: 0.875
Top Contributing Words:
- waiter (contribution: 0.0755)
- food (contribution: 0.0705)
- dame (contribution: 0.0504)
- crepe (contribution: 0.0491)
- notre (contribution: 0.0476)

Recommendation 3: 1755379
Similarity Score: 0.872
Top Contributing Words:
- food (contribution: 0.0694)
- waiter (contribution: 0.0643)
- service (contribution: 0.0445)
- good (contribution: 0.0394)
- place (contribution: 0.0308)

Recommendation 4: 2239316
Similarity Score: 0.850
Top Contributing Words:
- waiter (contribution: 0.0714)
- dame (contribution: 0.0531)
- notre (contribution: 0.0509)
- food (contribution: 0.0497)
- order (contribution: 0.0455)

Recommendation 5: 2337904
Similarity Score: 0.822
Top Contri

In [24]:
ids_to_keep = [1725986 , 2087229, 5924082]
tripadvisor_test = tripadvisor[tripadvisor['id'].isin(ids_to_keep)]
tripadvisor_test

,id,nom,url,website,rating,nbAvis,latitude,longitude,typeR,adresse,priceRange,activiteType,activiteSubType,activiteCategory,restaurantType,restaurantCategory,restaurantCuisine,restaurantDietaryRestrictions,hotelType,hotelpriceRange
261,1725986,Le Soleil D'Or,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d1725986-Reviews-Le_Soleil_D_Or-Paris_Ile_de_France.html,http://www.facebook.com/pages/Le-Soleil-Dor/237135486415057,3.134947,387,48.854130,2.345115,R,"15 boulevard du Palais, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, Cafe, European",NaN,NaN,NaN
301,2087229,Le Parvis,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d2087229-Reviews-Le_Parvis-Paris_Ile_de_France.html,http://www.restaurantleparvis.fr/,3.361972,309,48.854770,2.350077,R,"3 rue d Arcole, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, Bar, European",NaN,NaN,NaN
515,5924082,Brasserie Esmeralda,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d5924082-Reviews-Brasserie_Esmeralda-Paris_Ile_de_France.html,NaN,2.740275,379,48.852695,2.352112,R,"2 rue du Cloitre Notre Dame, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, European",NaN,NaN,NaN


In [25]:
tripadvisor_meta = tripadvisor.set_index("id")

evaluation_level1_m03 = evaluation_level1(X_test03, X_train03, similarity_matrix_m03, tripadvisor, top_k=5)
print(f"Precision Level 1 (Normalized Top-5 Type Score) : {evaluation_level1_m03:.4f}")

evaluation_level2_m03 = evaluation_level2(X_test03, X_train03, similarity_matrix_m03, tripadvisor_meta, top_k=5)
print(f"Precision Level 2 (Normalized Top-5 Metadata Score) : {evaluation_level2_m03:.4f}")

evaluation_level3_m03 = evaluation_level3(X_test03, X_train03, similarity_matrix_m03, tripadvisor)
print(f"Precision Level 3 (Top-1 Type Accuracy) : {evaluation_level3_m03:.4f}")

evaluation_level4_m03 = evaluation_level4(X_test03, X_train03, similarity_matrix_m03, tripadvisor_meta)
print(f"Precision Level 4 (Top-1 Metadata Accuracy) : {evaluation_level4_m03:.4f}")

evaluation_level5_m03 = evaluation_level5(X_test03, X_train03, similarity_matrix_m03, tripadvisor, top_k=5)
print(f"Precision Level 5 (Top-5 Type Recall) : {evaluation_level5_m03:.4f}")

evaluation_level6_m03 = evaluation_level6(X_test03, X_train03, similarity_matrix_m03, tripadvisor_meta, top_k=5)
print(f"Precision Level 6 (Top-5 Metadata Recall) : {evaluation_level6_m03:.4f}")

evaluation_level7_m03 = evaluation_level7(X_test03, X_train03, similarity_matrix_m03, tripadvisor)
print(f"Precision Level 7 (Mean Rank of First Relevant Type Match) : {evaluation_level7_m03:.2f}")

evaluation_level8_m03 = evaluation_level8(X_test03, X_train03, similarity_matrix_m03, tripadvisor_meta)
print(f"Precision Level 8 (Mean Rank of First Relevant Metadata Match) : {evaluation_level8_m03:.2f}")

Precision Level 1 (Normalized Top-5 Type Score) : 0.8447
Precision Level 2 (Normalized Top-5 Metadata Score) : 0.6872
Precision Level 3 (Top-1 Type Accuracy) : 0.8706
Precision Level 4 (Top-1 Metadata Accuracy) : 0.7310
Precision Level 5 (Top-5 Type Recall) : 0.9765
Precision Level 6 (Top-5 Metadata Recall) : 0.7176
Precision Level 7 (Mean Rank of First Relevant Type Match) : 1.52
Precision Level 8 (Mean Rank of First Relevant Metadata Match) : 42.66


Method 3 evaluates retrieval quality using TF-IDF representation and cosine similarity ranking. The system is evaluated using Precision@K by verifying whether retrieved places share the same typeR category as the query place.

## TF-IDF + NLP Pretreatments (NLTK)

In [26]:
reviews['cleaned_review'] = reviews['review'].apply(preprocess_text)
grouped = reviews.groupby('idplace')['cleaned_review'].apply(lambda x: " ".join(x)).reset_index()

# ===================================
# Mapping metadata
# ===================================
id_to_typeR = tripadvisor.set_index('id')['typeR'].to_dict()
grouped['typeR'] = grouped['idplace'].map(id_to_typeR)

# ===================================
# Train / Test Split
# ===================================
X_train04, X_test04 = train_test_split(
    grouped,
    test_size=0.1,
    random_state=42
)
X_train04 = X_train04.reset_index(drop=True)
X_test04 = X_test04.reset_index(drop=True)

# ===================================
# Improved TF-IDF Representation
# ===================================
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),   # add bigrams
    min_df=2
)
X_train_tfidf = vectorizer.fit_transform(X_train04['cleaned_review'])
X_test_tfidf = vectorizer.transform(X_test04['cleaned_review'])

# ===================================
# Cosine Similarity
# ===================================
similarity_matrix_m04 = cosine_similarity(X_test_tfidf, X_train_tfidf)

# ===================================
# Precision@K Evaluation
# ===================================
def precision_at_k(sim_matrix, train_labels, test_labels, k=10):
    correct = 0
    total = len(test_labels)
    for i in range(total):
        sims = sim_matrix[i]
        top_k = sims.argsort()[::-1][:k]
        if test_labels[i] in train_labels[top_k]:
            correct += 1

    return correct / total

train_categories = X_train04['typeR'].fillna("UNKNOWN").values
test_categories = X_test04['typeR'].fillna("UNKNOWN").values

K = 10
precision_level04 = precision_at_k(
    similarity_matrix_m04,
    train_categories,
    test_categories,
    k=K
)
print(f"\nPrecision@{K} Method 4 (TF-IDF + NLP preprocessing) : {precision_level04:.4f}")


Precision@10 Method 4 (TF-IDF + NLP preprocessing) : 0.9882


In [27]:
def recommend_similar_place_nlp_pretreatment(idplace_query, top_k=5, k_words=5):

    # Vérifier existence
    row = grouped[grouped['idplace'] == int(idplace_query)]
    if len(row) == 0:
        return "Idplace non trouvé dans le dataset"

    # --------------------------
    # Préparer texte requête
    # --------------------------
    raw_text = row.iloc[0]['cleaned_review']
    processed_text = preprocess_text(raw_text)

    # Transformer requête
    query_vector = vectorizer.transform([processed_text])

    # Similarité avec train
    sims = cosine_similarity(query_vector, X_train_tfidf)[0]

    # Trier les similarités décroissantes
    top_indices = np.argsort(sims)[::-1][:top_k]

    feature_names = vectorizer.get_feature_names_out()

    print(f"Query Place: {idplace_query}\n")

    for rank, idx in enumerate(top_indices, start=1):

        score = sims[idx]
        recommended_id = X_train04.iloc[idx]['idplace']

        # --------------------------
        # Explication des mots
        # --------------------------
        query_vec_dense = query_vector.toarray()[0]
        train_vec_dense = X_train_tfidf[idx].toarray()[0]

        contributions = query_vec_dense * train_vec_dense
        top_word_indices = np.argsort(contributions)[::-1][:k_words]

        print(f"Recommendation {rank}: {recommended_id}")
        print(f"Similarity Score: {score:.3f}")
        print("Top Contributing Words:")

        for word_idx in top_word_indices:
            if contributions[word_idx] > 0:
                print(
                    f"- {feature_names[word_idx]} "
                    f"(contribution: {contributions[word_idx]:.4f})"
                )
        print()

    return [int(X_train04.iloc[idx]['idplace']) for idx in top_indices]

id_to_test = 1725986
print(recommend_similar_place_nlp_pretreatment(id_to_test))

Query Place: 1725986

Recommendation 1: 5924082
Similarity Score: 0.845
Top Contributing Words:
- crepe (contribution: 0.0740)
- waiter (contribution: 0.0581)
- notre (contribution: 0.0376)
- dame (contribution: 0.0376)
- food (contribution: 0.0366)

Recommendation 2: 2087229
Similarity Score: 0.827
Top Contributing Words:
- waiter (contribution: 0.0574)
- food (contribution: 0.0536)
- dame (contribution: 0.0383)
- crepe (contribution: 0.0373)
- notre (contribution: 0.0362)

Recommendation 3: 2239316
Similarity Score: 0.817
Top Contributing Words:
- waiter (contribution: 0.0531)
- dame (contribution: 0.0395)
- notre (contribution: 0.0378)
- food (contribution: 0.0370)
- notre dame (contribution: 0.0366)

Recommendation 4: 1755379
Similarity Score: 0.811
Top Contributing Words:
- food (contribution: 0.0505)
- waiter (contribution: 0.0468)
- service (contribution: 0.0324)
- good (contribution: 0.0287)
- place (contribution: 0.0224)

Recommendation 5: 2337904
Similarity Score: 0.761
Top C

In [28]:
ids_to_keep = [1725986 , 2087229, 5924082, 1755379]
tripadvisor_test = tripadvisor[tripadvisor['id'].isin(ids_to_keep)]
tripadvisor_test

,id,nom,url,website,rating,nbAvis,latitude,longitude,typeR,adresse,priceRange,activiteType,activiteSubType,activiteCategory,restaurantType,restaurantCategory,restaurantCuisine,restaurantDietaryRestrictions,hotelType,hotelpriceRange
261,1725986,Le Soleil D'Or,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d1725986-Reviews-Le_Soleil_D_Or-Paris_Ile_de_France.html,http://www.facebook.com/pages/Le-Soleil-Dor/237135486415057,3.134947,387,48.854130,2.345115,R,"15 boulevard du Palais, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, Cafe, European",NaN,NaN,NaN
263,1755379,Brasserie Les Deux Palais,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d1755379-Reviews-Brasserie_Les_Deux_Palais-Paris_Ile_de_France.html,http://www.brasserielesdeuxpalais.fr/,3.947851,969,48.855370,2.346088,R,"3 boulevard du Palais, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, Bar, European, Vegetarian Friendly, Gluten Free Options","Vegetarian Friendly, Gluten Free Options",NaN,NaN
301,2087229,Le Parvis,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d2087229-Reviews-Le_Parvis-Paris_Ile_de_France.html,http://www.restaurantleparvis.fr/,3.361972,309,48.854770,2.350077,R,"3 rue d Arcole, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, Bar, European",NaN,NaN,NaN
515,5924082,Brasserie Esmeralda,https://www.tripadvisor.co.uk/Restaurant_Review-g187147-d5924082-Reviews-Brasserie_Esmeralda-Paris_Ile_de_France.html,NaN,2.740275,379,48.852695,2.352112,R,"2 rue du Cloitre Notre Dame, 75004 Paris France",€€€,NaN,NaN,NaN,Restaurants,sit_down,"French, European",NaN,NaN,NaN


In [29]:
tripadvisor_meta = tripadvisor.set_index("id")

evaluation_level1_m04 = evaluation_level1(X_test04, X_train04, similarity_matrix_m04, tripadvisor, top_k=5)
print(f"Precision Level 1 (Normalized Top-5 Type Score) : {evaluation_level1_m04:.4f}")

evaluation_level2_m04 = evaluation_level2(X_test04, X_train04, similarity_matrix_m04, tripadvisor_meta, top_k=5)
print(f"Precision Level 2 (Normalized Top-5 Metadata Score) : {evaluation_level2_m04:.4f}")

evaluation_level3_m04 = evaluation_level3(X_test04, X_train04, similarity_matrix_m04, tripadvisor)
print(f"Precision Level 3 (Top-1 Type Accuracy) : {evaluation_level3_m04:.4f}")

evaluation_level4_m04 = evaluation_level4(X_test04, X_train04, similarity_matrix_m04, tripadvisor_meta)
print(f"Precision Level 4 (Top-1 Metadata Accuracy) : {evaluation_level4_m04:.4f}")

evaluation_level5_m04 = evaluation_level5(X_test04, X_train04, similarity_matrix_m04, tripadvisor, top_k=5)
print(f"Precision Level 5 (Top-5 Type Recall) : {evaluation_level5_m04:.4f}")

evaluation_level6_m04 = evaluation_level6(X_test04, X_train04, similarity_matrix_m04, tripadvisor_meta, top_k=5)
print(f"Precision Level 6 (Top-5 Metadata Recall) : {evaluation_level6_m04:.4f}")

evaluation_level7_m04 = evaluation_level7(X_test04, X_train04, similarity_matrix_m04, tripadvisor)
print(f"Precision Level 7 (Mean Rank of First Relevant Type Match) : {evaluation_level7_m04:.2f}")

evaluation_level8_m04 = evaluation_level8(X_test04, X_train04, similarity_matrix_m04, tripadvisor_meta)
print(f"Precision Level 8 (Mean Rank of First Relevant Metadata Match) : {evaluation_level8_m04:.2f}")

Precision Level 1 (Normalized Top-5 Type Score) : 0.8518
Precision Level 2 (Normalized Top-5 Metadata Score) : 0.6874
Precision Level 3 (Top-1 Type Accuracy) : 0.8824
Precision Level 4 (Top-1 Metadata Accuracy) : 0.7359
Precision Level 5 (Top-5 Type Recall) : 0.9765
Precision Level 6 (Top-5 Metadata Recall) : 0.6353
Precision Level 7 (Mean Rank of First Relevant Type Match) : 1.33
Precision Level 8 (Mean Rank of First Relevant Metadata Match) : 48.38
